# External-mouse-only RatLesNetV2 and nnU-Net training — Kaggle P100

This notebook trains exactly two source-domain models using only the 426 human-reviewed cases in `External_Mouse_T2w_manual_LSP_SI_v0`:

1. RatLesNetV2 initialized from the pinned upstream rat checkpoint and fine-tuned with CE+Dice.
2. nnU-Net v2 ResEnc-M initialized from scratch and trained with `nnUNetTrainer_250epochs`.

Both models use the same deterministic 341/85 train/validation split grouped by manifest `animal_id` (seed `20260715`). There is no test partition, no LYS target dataset, no target fine-tuning, no OOF threshold selection, and no locked-test inference. External validation is source-training evidence only and must not be presented as LYS performance. Predictions remain draft masks.

Enable a Kaggle P100 GPU and Internet. Attach only the private `External_Mouse_T2w_manual_LSP_SI_v0` Kaggle dataset. The input may be either the original `.tar.gz` or Kaggle's exposed directory tree.


## 1 — Exact software and training configuration

CE+Dice is fixed here as an explicit standalone source-training configuration; it is not selected using external validation. RatLesNetV2 uses at most 50 epochs, validation every epoch, best checkpoint by validation Dice, early stopping 12, and reduce-LR patience 4. nnU-Net uses the pinned v2.8.1 source revision, official ResEnc-M planner, one preserved fold, and the official 250-epoch budget trainer. Do not change the trainer after a run has started.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import tarfile
import zipfile
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

RUN_SEED = 20260715
BRANCH = "dl-ratlesnetv2-finetune"
REPOSITORY = "https://github.com/paulaize/LYS_PROJ1.git"
RATLESNET_COMMIT = "c9dfb7eddf5c3369151d2f7a64add9a42b2d6983"
NNUNET_COMMIT = "468cf803df9b267150ae2b6c0c59b8ac84f16227"  # v2.8.1
EXTERNAL_NAME = "External_Mouse_T2w_manual_LSP_SI_v0"
EXPECTED_CASES = 426
EXPECTED_TRAIN = 341
EXPECTED_VALIDATION = 85
RATLES_LOSS = "ce-dice"
RATLES_LR = 5e-5
RATLES_EPOCHS = 50
NNUNET_ID = 702
NNUNET_DATASET = "Dataset702_ExternalMouseV0"
NNUNET_TRAINER = "nnUNetTrainer_250epochs"
NNUNET_PLANNER = "nnUNetPlannerResEncM"
NNUNET_PLANS = "nnUNetResEncUNetMPlans"
NNUNET_CONFIGURATION = "3d_fullres"
RESUME_ARCHIVE_OVERRIDE = None  # optional absolute Kaggle input path

WORK = Path("/kaggle/working")
PROJECT = WORK / "LYS_PROJ1"
if not (PROJECT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only"], check=True)
PROJECT_COMMIT = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT / "ratlesnetv2_finetune/requirements-colab.txt"),
        f"git+https://github.com/MIC-DKFZ/nnUNet.git@{NNUNET_COMMIT}",
        "matplotlib",
    ],
    check=True,
)
sys.path.insert(0, str(PROJECT))

assert importlib.metadata.version("nnunetv2") == "2.8.1"
assert torch.cuda.is_available(), "Enable a Kaggle GPU before continuing"
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if "P100" not in GPU_NAME.upper():
    print("WARNING: ResEnc-M was budgeted for a P100; provenance records this GPU.")
subprocess.run(["nvidia-smi"], check=True)

for module in (
    "ratlesnetv2_finetune.scripts.normalize_prepared_dataset",
    "ratlesnetv2_finetune.scripts.split_prepared_dataset",
    "ratlesnetv2_finetune.scripts.finetune_ratlesnetv2",
    "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
):
    subprocess.run(
        [sys.executable, "-m", module, "--help"],
        cwd=PROJECT,
        check=True,
        stdout=subprocess.DEVNULL,
    )
for command in ("nnUNetv2_plan_and_preprocess", "nnUNetv2_train"):
    assert shutil.which(command), command
    subprocess.run([command, "--help"], check=True, stdout=subprocess.DEVNULL)

RAT_REPO = WORK / "RatLesNetv2"
if not (RAT_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/jmlipman/RatLesNetv2.git", str(RAT_REPO)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(RAT_REPO), "fetch", "--depth", "1",
     "origin", RATLESNET_COMMIT],
    check=True,
)
subprocess.run(
    ["git", "-C", str(RAT_REPO), "checkout", "--detach", RATLESNET_COMMIT],
    check=True,
)
resolved_ratlesnet_commit = subprocess.check_output(
    ["git", "-C", str(RAT_REPO), "rev-parse", "HEAD"], text=True
).strip()
assert resolved_ratlesnet_commit == RATLESNET_COMMIT
RAT_PRETRAINED = RAT_REPO / "trained_models/Table2-3/RatLesNetv2/homogeneous/model-1"
assert RAT_PRETRAINED.is_file(), RAT_PRETRAINED

NNUNET_RAW = WORK / "nnUNet_raw"
NNUNET_PREPROCESSED = WORK / "nnUNet_preprocessed"
NNUNET_RESULTS = WORK / "nnUNet_results"
for key, value in {
    "nnUNet_raw": NNUNET_RAW,
    "nnUNet_preprocessed": NNUNET_PREPROCESSED,
    "nnUNet_results": NNUNET_RESULTS,
}.items():
    os.environ[key] = str(value)

EXPERIMENT_ROOT = WORK / "external_mouse_ratlesnetv2_nnunet"
PROVENANCE = EXPERIMENT_ROOT / "provenance"
RUNS = EXPERIMENT_ROOT / "runs"
print("Project commit:", PROJECT_COMMIT)
print("RatLesNetV2 commit:", resolved_ratlesnet_commit)
print("nnU-Net commit:", NNUNET_COMMIT)


## 2 — Restore a prior training session (optional)

Attach at most one previous `External_Mouse_RatLesNetV2_nnUNet_resume.tar.gz`. The archive contains checkpoints and reports, never raw images. Normalization, the split, nnU-Net conversion, and preprocessing are rebuilt and verified on every new Kaggle session.


In [ ]:
def safe_extract(archive_path: Path, destination: Path) -> None:
    root = destination.resolve()
    with tarfile.open(archive_path, "r:*") as archive:
        for member in archive.getmembers():
            target = (destination / member.name).resolve()
            assert target == root or root in target.parents, member.name
        archive.extractall(destination)


if RESUME_ARCHIVE_OVERRIDE:
    resume_archives = [Path(RESUME_ARCHIVE_OVERRIDE)]
else:
    resume_archives = sorted(
        Path("/kaggle/input").rglob(
            "External_Mouse_RatLesNetV2_nnUNet_resume.tar.gz"
        )
    )
assert len(resume_archives) <= 1, resume_archives
if resume_archives:
    assert not EXPERIMENT_ROOT.exists() and not NNUNET_RESULTS.exists(), (
        "Refusing to merge a resume archive into existing run state"
    )
    safe_extract(resume_archives[0], WORK)
    print("Restored:", resume_archives[0])
else:
    print("No resume archive attached.")

for path in (EXPERIMENT_ROOT, PROVENANCE, RUNS):
    path.mkdir(parents=True, exist_ok=True)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def build_resume_archive() -> Path:
    output = WORK / "External_Mouse_RatLesNetV2_nnUNet_resume.tar.gz"
    with tarfile.open(output, "w:gz", compresslevel=1) as archive:
        archive.add(EXPERIMENT_ROOT, arcname=EXPERIMENT_ROOT.relative_to(WORK))
        if NNUNET_RESULTS.exists():
            archive.add(NNUNET_RESULTS, arcname=NNUNET_RESULTS.relative_to(WORK))
    return output


## 3 — Locate and normalize only the external dataset

The cell fails if a LYS prepared dataset or archive is attached. The payload-aware normalizer detects gzip from the bytes, fully loads every NIfTI, validates scan/mask geometry and both mask aliases, and writes canonical files only under `/kaggle/working`.


In [ ]:
INPUT_ROOT = Path("/kaggle/input")
forbidden_target_inputs = sorted(INPUT_ROOT.rglob("LYS_T2w_manual_v1.tar.gz"))
forbidden_target_inputs += sorted(INPUT_ROOT.rglob("LYS_T2w_manual_v1/manifest.csv"))
assert not forbidden_target_inputs, (
    "Detach the LYS target dataset before running this external-only notebook: "
    f"{forbidden_target_inputs}"
)


def locate_or_extract_prepared_dataset(dataset_name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(f"{dataset_name}/manifest.csv"))
    if len(matches) == 1:
        return matches[0].parent
    assert not matches, f"Multiple exposed {dataset_name} roots: {matches}"
    archives = sorted(INPUT_ROOT.rglob(f"{dataset_name}.tar.gz"))
    assert len(archives) == 1, (
        f"Expected exactly one {dataset_name} input; found {archives}"
    )
    extract_root = WORK / "uploaded_archives" / dataset_name
    extract_root.mkdir(parents=True, exist_ok=True)
    safe_extract(archives[0], extract_root)
    matches = sorted(extract_root.rglob(f"{dataset_name}/manifest.csv"))
    assert len(matches) == 1, matches
    return matches[0].parent


EXTERNAL_SOURCE = locate_or_extract_prepared_dataset(EXTERNAL_NAME)
NORMALIZED_BASE = WORK / "normalized_inputs"
subprocess.run(
    [
        sys.executable,
        "-m",
        "ratlesnetv2_finetune.scripts.normalize_prepared_dataset",
        "--input",
        str(EXTERNAL_SOURCE),
        "--output-base",
        str(NORMALIZED_BASE),
        "--dataset-name",
        EXTERNAL_NAME,
        "--expected-cases",
        str(EXPECTED_CASES),
    ],
    cwd=PROJECT,
    check=True,
)
EXTERNAL_ROOT = NORMALIZED_BASE / EXTERNAL_NAME
assert (EXTERNAL_ROOT / "manifest.csv").is_file()
assert (EXTERNAL_ROOT / "normalization_report.csv").is_file()
assert len(list(EXTERNAL_ROOT.rglob("scan.nii.gz"))) == EXPECTED_CASES
print("Normalized external root:", EXTERNAL_ROOT)


## 4 — Create and lock the shared 341/85 external split

There is deliberately no external test partition. The validation subset selects each source checkpoint and reports source-domain metrics; it is not target-domain evidence.


In [ ]:
EXTERNAL_SPLIT = WORK / "External_pretrain_split"
subprocess.run(
    [
        sys.executable,
        "-m",
        "ratlesnetv2_finetune.scripts.split_prepared_dataset",
        "--input",
        str(EXTERNAL_ROOT),
        "--output",
        str(EXTERNAL_SPLIT),
        "--group-by",
        "animal_id",
        "--validation-fraction",
        "0.20",
        "--test-fraction",
        "0",
        "--seed",
        str(RUN_SEED),
        "--copy-mode",
        "symlink",
        "--overwrite",
    ],
    cwd=PROJECT,
    check=True,
)
external_manifest = pd.read_csv(EXTERNAL_SPLIT / "manifest.csv", keep_default_na=False)
assert len(external_manifest) == EXPECTED_CASES
assert external_manifest.case_id.is_unique
assert dict(external_manifest.split.value_counts()) == {
    "train": EXPECTED_TRAIN,
    "validation": EXPECTED_VALIDATION,
}
assert not (
    set(external_manifest.loc[external_manifest.split == "train", "animal_id"])
    & set(
        external_manifest.loc[external_manifest.split == "validation", "animal_id"]
    )
)
assert len(list((EXTERNAL_SPLIT / "train").rglob("scan.nii.gz"))) == EXPECTED_TRAIN
assert (
    len(list((EXTERNAL_SPLIT / "validation").rglob("scan.nii.gz")))
    == EXPECTED_VALIDATION
)
assert not (EXTERNAL_SPLIT / "test").exists() or not list(
    (EXTERNAL_SPLIT / "test").rglob("scan.nii.gz")
)
external_manifest["source_dataset"] = (
    external_manifest.case_id.str.split("__").str[0]
)
source_counts = pd.crosstab(external_manifest.split, external_manifest.source_dataset)
expected_source_counts = {
    "train": {"An2022": 265, "Knab2025": 66, "Koch2017": 10},
    "validation": {"An2022": 66, "Knab2025": 14, "Koch2017": 5},
}
observed_source_counts = {
    split: {source: int(source_counts.loc[split, source]) for source in source_counts.columns}
    for split in source_counts.index
}
assert observed_source_counts == expected_source_counts, observed_source_counts
display(source_counts)

for source, destination in (
    (EXTERNAL_ROOT / "manifest.csv", PROVENANCE / "normalized_manifest.csv"),
    (
        EXTERNAL_ROOT / "normalization_report.csv",
        PROVENANCE / "normalization_report.csv",
    ),
    (EXTERNAL_SPLIT / "manifest.csv", PROVENANCE / "split_manifest.csv"),
    (EXTERNAL_SPLIT / "split_summary.json", PROVENANCE / "split_summary.json"),
):
    shutil.copy2(source, destination)

protocol_identity = {
    "protocol": "external_mouse_source_training_v1",
    "project_commit": PROJECT_COMMIT,
    "ratlesnet_source_commit": RATLESNET_COMMIT,
    "ratlesnet_upstream_checkpoint": str(RAT_PRETRAINED.relative_to(RAT_REPO)),
    "ratlesnet_upstream_checkpoint_sha256": sha256(RAT_PRETRAINED),
    "nnunet_source_commit": NNUNET_COMMIT,
    "nnunet_version": importlib.metadata.version("nnunetv2"),
    "run_seed": RUN_SEED,
    "gpu_name": GPU_NAME,
    "dataset_name": EXTERNAL_NAME,
    "dataset_cases": EXPECTED_CASES,
    "normalized_manifest_sha256": sha256(EXTERNAL_ROOT / "manifest.csv"),
    "split_manifest_sha256": sha256(EXTERNAL_SPLIT / "manifest.csv"),
    "train_cases": EXPECTED_TRAIN,
    "validation_cases": EXPECTED_VALIDATION,
    "test_cases": 0,
    "split_group": "animal_id",
    "ratlesnet_initialization": "pinned_upstream_rat_checkpoint",
    "ratlesnet_loss": RATLES_LOSS,
    "ratlesnet_learning_rate": RATLES_LR,
    "ratlesnet_max_epochs": RATLES_EPOCHS,
    "nnunet_trainer": NNUNET_TRAINER,
    "nnunet_planner": NNUNET_PLANNER,
    "nnunet_plans": NNUNET_PLANS,
    "nnunet_configuration": NNUNET_CONFIGURATION,
    "nnunet_initialization": "from_scratch",
    "nnunet_canonical_1000_epoch_claim": False,
    "lys_data_used": False,
    "postprocessing": "none",
}
identity_path = PROVENANCE / "protocol_identity.json"
if identity_path.is_file():
    assert json.loads(identity_path.read_text()) == protocol_identity, (
        "Restored outputs use a different protocol or software identity"
    )
else:
    identity_path.write_text(
        json.dumps(protocol_identity, indent=2, sort_keys=True) + "\n"
    )
display(protocol_identity)


## 5 — Train RatLesNetV2 on external train only

The pinned upstream model is initialization, not an input dataset. All supervised optimization and checkpoint selection performed here use only the external train/validation split. Final validation probability maps are exported from the restored best-by-validation-Dice checkpoint.


In [ ]:
RATLES_OUTPUT = RUNS / "ratlesnetv2_external_ce_dice"


def successful_ratles_run(output_root: Path) -> Path | None:
    if not output_root.is_dir():
        return None
    candidates = sorted(
        (path for path in output_root.iterdir() if path.is_dir() and path.name.isdigit()),
        key=lambda path: int(path.name),
        reverse=True,
    )
    required_names = {
        "RatLesNetv2.model",
        "run_status.json",
        "run_config.json",
        "selected_checkpoint.json",
        "final_metrics.json",
    }
    for run in candidates:
        if not all((run / name).is_file() for name in required_names):
            continue
        status = json.loads((run / "run_status.json").read_text())["status"]
        prediction_manifest = (
            run
            / "prediction_exports"
            / "validation"
            / "final"
            / "prediction_export_manifest.csv"
        )
        if status in {"completed", "early_stopped"} and prediction_manifest.is_file():
            return run
    return None


ratles_run = successful_ratles_run(RATLES_OUTPUT)
if ratles_run is None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "ratlesnetv2_finetune.scripts.finetune_ratlesnetv2",
            "--ratlesnet-repo",
            str(RAT_REPO),
            "--input",
            str(EXTERNAL_SPLIT / "train"),
            "--validation",
            str(EXTERNAL_SPLIT / "validation"),
            "--output",
            str(RATLES_OUTPUT),
            "--pretrained-model",
            str(RAT_PRETRAINED),
            "--require-pretrained",
            "--loss",
            RATLES_LOSS,
            "--lr",
            str(RATLES_LR),
            "--epochs",
            str(RATLES_EPOCHS),
            "--seed",
            str(RUN_SEED),
            "--gpu",
            "0",
            "--loadMemory",
            "0",
            "--save-every",
            "0",
            "--eval-every",
            "1",
            "--metrics-threshold",
            "0.5",
            "--early-stop-patience",
            "12",
            "--early-stop-min-delta",
            "0.001",
            "--lr-scheduler",
            "reduce-on-plateau",
            "--lr-scheduler-metric",
            "validation_dice",
            "--lr-plateau-patience",
            "4",
            "--lr-plateau-factor",
            "0.5",
            "--lr-plateau-min-delta",
            "0.001",
            "--min-lr",
            "1e-6",
            "--export-predictions",
            "validation",
            "--export-prediction-limit",
            "0",
            "--export-prediction-epochs",
            "final",
        ],
        cwd=PROJECT,
        check=True,
    )
    ratles_run = successful_ratles_run(RATLES_OUTPUT)
assert ratles_run is not None
ratles_probability_manifest = (
    ratles_run
    / "prediction_exports"
    / "validation"
    / "final"
    / "prediction_export_manifest.csv"
)
ratles_predictions = pd.read_csv(ratles_probability_manifest)
expected_validation_case_ids = set(
    external_manifest.loc[external_manifest.split == "validation", "case_id"]
)
assert len(ratles_predictions) == EXPECTED_VALIDATION
assert ratles_predictions.case_id.is_unique
assert set(ratles_predictions.case_id) == expected_validation_case_ids
ratles_record = {
    "status": json.loads((ratles_run / "run_status.json").read_text())["status"],
    "run_directory": str(ratles_run.relative_to(WORK)),
    "selected_checkpoint": str((ratles_run / "RatLesNetv2.model").relative_to(WORK)),
    "selected_checkpoint_sha256": sha256(ratles_run / "RatLesNetv2.model"),
    "checkpoint_selection": "best external-validation Dice",
    "validation_metrics": str((ratles_run / "final_metrics.json").relative_to(WORK)),
    "validation_probability_manifest": str(
        ratles_probability_manifest.relative_to(WORK)
    ),
    "validation_cases": EXPECTED_VALIDATION,
    "lys_data_used": False,
}
(PROVENANCE / "ratlesnetv2_training_record.json").write_text(
    json.dumps(ratles_record, indent=2, sort_keys=True) + "\n"
)
display(ratles_record)
display(json.loads((ratles_run / "final_metrics.json").read_text()))


In [ ]:
interim_archive = build_resume_archive()
print("RatLesNetV2-safe resume artifact:", interim_archive)
display(FileLink(str(interim_archive)))


## 6 — Convert and preprocess the same external split for nnU-Net

All 426 external cases are converted to nnU-Net's supervised `imagesTr`/`labelsTr` contract. The repository writes the exact shared 341/85 assignment as a one-fold `splits_final.json`; the notebook installs that file after preprocessing so nnU-Net cannot invent another split. Planning and preprocessing use only this external dataset.


In [ ]:
EXTERNAL_RAW = NNUNET_RAW / NNUNET_DATASET
subprocess.run(
    [
        sys.executable,
        "-m",
        "ratlesnetv2_finetune.scripts.prepare_architecture_comparator",
        "prepare-external",
        "--input",
        str(EXTERNAL_SPLIT),
        "--output",
        str(EXTERNAL_RAW),
        "--dataset-id",
        str(NNUNET_ID),
        "--dataset-name",
        "ExternalMouseV0",
    ],
    cwd=PROJECT,
    check=True,
)
conversion_report = json.loads((EXTERNAL_RAW / "conversion_report.json").read_text())
assert conversion_report["n_training"] == EXPECTED_CASES
assert conversion_report["source_split_counts"] == {
    "train": EXPECTED_TRAIN,
    "validation": EXPECTED_VALIDATION,
}
assert conversion_report["target_lys_used"] is False
external_mapping = pd.read_csv(EXTERNAL_RAW / "case_mapping.csv", keep_default_na=False)
assert len(external_mapping) == EXPECTED_CASES
assert external_mapping.case_id.is_unique
assert set(external_mapping.case_id) == set(external_manifest.case_id)

EXTERNAL_PREPROCESSED = NNUNET_PREPROCESSED / NNUNET_DATASET
plan_path = EXTERNAL_PREPROCESSED / f"{NNUNET_PLANS}.json"
preprocess_marker = PROVENANCE / "nnunet_preprocessing_complete.json"
previous_plan_hash = (
    json.loads(preprocess_marker.read_text())["plans_sha256"]
    if preprocess_marker.is_file()
    else None
)
if not plan_path.is_file():
    subprocess.run(
        [
            "nnUNetv2_plan_and_preprocess",
            "-d",
            str(NNUNET_ID),
            "-pl",
            NNUNET_PLANNER,
            "-c",
            NNUNET_CONFIGURATION,
            "-npfp",
            "4",
            "-np",
            "4",
            "--verify_dataset_integrity",
        ],
        check=True,
    )
assert plan_path.is_file(), plan_path
if previous_plan_hash is not None:
    assert sha256(plan_path) == previous_plan_hash, (
        "Rebuilt nnU-Net plans differ from the resumed training session"
    )
preprocess_marker.write_text(
    json.dumps({"plans_sha256": sha256(plan_path)}, indent=2, sort_keys=True) + "\n"
)
shutil.copy2(
    EXTERNAL_RAW / "splits_final.json",
    EXTERNAL_PREPROCESSED / "splits_final.json",
)
installed_split = json.loads(
    (EXTERNAL_PREPROCESSED / "splits_final.json").read_text()
)
raw_split = json.loads((EXTERNAL_RAW / "splits_final.json").read_text())
assert installed_split == raw_split and len(installed_split) == 1
assert len(installed_split[0]["train"]) == EXPECTED_TRAIN
assert len(installed_split[0]["val"]) == EXPECTED_VALIDATION
assert not set(installed_split[0]["train"]) & set(installed_split[0]["val"])

for name in ("conversion_report.json", "case_mapping.csv", "splits_final.json"):
    shutil.copy2(EXTERNAL_RAW / name, PROVENANCE / f"nnunet_{name}")
shutil.copy2(plan_path, PROVENANCE / plan_path.name)
plans = json.loads(plan_path.read_text())
configuration = plans["configurations"][NNUNET_CONFIGURATION]
display(conversion_report)
display(
    {
        key: configuration.get(key)
        for key in (
            "data_identifier",
            "median_image_size_in_voxels",
            "spacing",
            "patch_size",
            "batch_size",
            "architecture",
        )
    }
)


## 7 — Train nnU-Net on external train only

Fold 0 is the one preserved external split. A partially completed nnU-Net run resumes with `--c`. A completed run re-exports validation from `checkpoint_best.pth` with probabilities. The validation gate requires exactly the expected 85 external case IDs. If you interrupt this cell, rerun Cell 12 before ending the Kaggle session to archive `checkpoint_latest.pth`.


In [ ]:
NNUNET_MODEL_FOLDER = (
    NNUNET_RESULTS
    / NNUNET_DATASET
    / f"{NNUNET_TRAINER}__{NNUNET_PLANS}__{NNUNET_CONFIGURATION}"
    / "fold_0"
)
NNUNET_BEST = NNUNET_MODEL_FOLDER / "checkpoint_best.pth"
NNUNET_FINAL = NNUNET_MODEL_FOLDER / "checkpoint_final.pth"
NNUNET_LATEST = NNUNET_MODEL_FOLDER / "checkpoint_latest.pth"
NNUNET_VALIDATION = NNUNET_MODEL_FOLDER / "validation"
NNUNET_STATUS = PROVENANCE / "nnunet_training_record.json"
NNUNET_PROBABILITY_MANIFEST = PROVENANCE / "nnunet_validation_probability_manifest.csv"
expected_nnunet_validation = set(
    external_mapping.loc[
        external_mapping.source_split == "validation", "nnunet_case_id"
    ]
)
assert len(expected_nnunet_validation) == EXPECTED_VALIDATION
base_command = [
    "nnUNetv2_train",
    str(NNUNET_ID),
    NNUNET_CONFIGURATION,
    "0",
    "-tr",
    NNUNET_TRAINER,
    "-p",
    NNUNET_PLANS,
]
if NNUNET_STATUS.is_file():
    recorded_nnunet = json.loads(NNUNET_STATUS.read_text())
    assert NNUNET_BEST.is_file()
    assert recorded_nnunet["selected_checkpoint_sha256"] == sha256(NNUNET_BEST)
elif NNUNET_FINAL.is_file():
    subprocess.run(base_command + ["--val", "--npz", "--val_best"], check=True)
else:
    train_command = base_command.copy()
    if NNUNET_LATEST.is_file():
        train_command.append("--c")
    train_command.extend(["--npz", "--val_best"])
    subprocess.run(train_command, check=True)

assert NNUNET_BEST.is_file(), NNUNET_BEST
assert NNUNET_VALIDATION.is_dir(), NNUNET_VALIDATION
observed_probability_ids = {path.stem for path in NNUNET_VALIDATION.glob("*.npz")}
assert observed_probability_ids == expected_nnunet_validation, (
    f"Missing={sorted(expected_nnunet_validation - observed_probability_ids)[:10]}, "
    f"unexpected={sorted(observed_probability_ids - expected_nnunet_validation)[:10]}"
)
validation_rows = []
validation_mapping = external_mapping[
    external_mapping.source_split == "validation"
].sort_values("case_id")
for row in validation_mapping.itertuples(index=False):
    probability = NNUNET_VALIDATION / f"{row.nnunet_case_id}.npz"
    segmentation = NNUNET_VALIDATION / f"{row.nnunet_case_id}.nii.gz"
    assert probability.is_file() and segmentation.is_file(), row.case_id
    validation_rows.append(
        {
            "split": "validation",
            "case_id": row.case_id,
            "nnunet_case_id": row.nnunet_case_id,
            "probability_npz": str(probability.relative_to(WORK)),
            "segmentation_nifti": str(segmentation.relative_to(WORK)),
            "checkpoint": "checkpoint_best.pth",
            "postprocessing": "none",
        }
    )
nnunet_probability_rows = pd.DataFrame(validation_rows)
assert len(nnunet_probability_rows) == EXPECTED_VALIDATION
assert nnunet_probability_rows.case_id.is_unique
assert set(nnunet_probability_rows.case_id) == expected_validation_case_ids
nnunet_probability_rows.to_csv(NNUNET_PROBABILITY_MANIFEST, index=False)
nnunet_summary_path = NNUNET_VALIDATION / "summary.json"
assert nnunet_summary_path.is_file(), nnunet_summary_path
recorded_nnunet = {
    "status": "completed",
    "model_directory": str(NNUNET_MODEL_FOLDER.relative_to(WORK)),
    "selected_checkpoint": str(NNUNET_BEST.relative_to(WORK)),
    "selected_checkpoint_sha256": sha256(NNUNET_BEST),
    "checkpoint_selection": "nnU-Net checkpoint_best; validation exported with --val_best",
    "validation_metrics": str(nnunet_summary_path.relative_to(WORK)),
    "validation_probability_manifest": str(
        NNUNET_PROBABILITY_MANIFEST.relative_to(WORK)
    ),
    "validation_cases": EXPECTED_VALIDATION,
    "trainer": NNUNET_TRAINER,
    "plans_sha256": sha256(plan_path),
    "lys_data_used": False,
}
NNUNET_STATUS.write_text(
    json.dumps(recorded_nnunet, indent=2, sort_keys=True) + "\n"
)
display(recorded_nnunet)
display(json.loads(nnunet_summary_path.read_text()))


## 8 — Verify completion and package checkpoints

The final summary keeps the models separate. Their validation reports are not a target-domain comparison, and the 250-epoch nnU-Net run is not a canonical 1,000-epoch claim. The full tarball contains both selected checkpoints, all training state needed for audit/resume, validation outputs, manifests, hashes, and provenance. It contains no raw input images.


In [ ]:
completion_summary = {
    "protocol": "external_mouse_source_training_v1",
    "dataset": EXTERNAL_NAME,
    "train_cases": EXPECTED_TRAIN,
    "validation_cases": EXPECTED_VALIDATION,
    "test_cases": 0,
    "ratlesnetv2": ratles_record,
    "nnunetv2": recorded_nnunet,
    "external_validation_is_lys_evidence": False,
    "architecture_selected": False,
    "probability_threshold_selected": False,
    "locked_test_used": False,
    "postprocessing": "none",
    "model_outputs_are_draft_masks": True,
}
summary_path = EXPERIMENT_ROOT / "training_completion_summary.json"
summary_path.write_text(
    json.dumps(completion_summary, indent=2, sort_keys=True) + "\n"
)
display(completion_summary)

REVIEW_ZIP = WORK / "External_Mouse_RatLesNetV2_nnUNet_review.zip"
allowed_suffixes = {".csv", ".json", ".md", ".png", ".txt", ".yaml", ".yml"}
with zipfile.ZipFile(
    REVIEW_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    roots = [EXPERIMENT_ROOT, EXTERNAL_RAW, EXTERNAL_PREPROCESSED, NNUNET_MODEL_FOLDER]
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("*")):
            if (
                path.is_file()
                and not path.is_symlink()
                and path.suffix.lower() in allowed_suffixes
                and path.stat().st_size <= 25 * 1024 * 1024
            ):
                archive.write(path, path.relative_to(WORK))
    notebook_path = PROJECT / "notebooks/external_mouse_ratlesnetv2_nnunet_kaggle.ipynb"
    if notebook_path.is_file():
        archive.write(
            notebook_path, Path("repository_snapshot") / notebook_path.relative_to(PROJECT)
        )
FULL_ARCHIVE = build_resume_archive()
print("Compact review bundle:", REVIEW_ZIP, f"{REVIEW_ZIP.stat().st_size / 1e6:.1f} MB")
print("Full checkpoint/resume bundle:", FULL_ARCHIVE, f"{FULL_ARCHIVE.stat().st_size / 1e9:.2f} GB")
display(FileLink(str(REVIEW_ZIP)))
display(FileLink(str(FULL_ARCHIVE)))


## End state

A complete run has one validation-selected RatLesNetV2 checkpoint, one validation-selected nnU-Net checkpoint, both validation metric reports, both probability manifests, the shared split manifest, normalization/conversion reports, software revisions, hashes, and the final artifact bundles. Do not add LYS fine-tuning, threshold calibration, or locked-test evaluation to this notebook.
